# Creating Your First LangChain Agent with Basic Tools

**Author:** Jack Pumpuni Frimpong-Manso ([GitHub: pumpuni07](https://github.com/pumpuni07))

Exercise 7 from the IBM Skills Network course **"Build Smarter AI Apps: Empower LLMs with LangChain"**, completed and extended for my AI/ML engineering portfolio.

## What this notebook builds

A **ReAct agent** (Reasoning + Acting) that decides *which tool to use and when* based on the user's question:

1. **Calculator** — safe arithmetic evaluation. The lab hint suggests Python's `eval()`, but `eval()` executes arbitrary code — a genuine security risk when the input comes from an LLM. This solution parses expressions into an **AST** and permits only arithmetic nodes, so injection attempts like `__import__('os')` are rejected.
2. **TextFormatter** — uppercase / lowercase / titlecase conversion with structured input parsing.

The agent follows the classic **ReAct loop**: *Thought → Action → Action Input → Observation → ... → Final Answer*.

## Version note (important)

This exercise uses the classic agent API (`create_react_agent`, `AgentExecutor`) from **LangChain 0.3.x**. These were **removed in LangChain 1.x**, where agents moved to a new `create_agent` API and LangGraph. `requirements.txt` pins the working versions. Verified against `langchain==0.3.27` and `langchain-ibm==0.3.15`.

## 1. Setup

In [ ]:
# Uncomment in a fresh environment — version pins matter here (see note above)
# %pip install --quiet langchain==0.3.27 langchain-ibm==0.3.15

In [ ]:
import ast
import operator
import os

from langchain_core.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

## 2. Tool 1 — Calculator (safe, AST-based)

Instead of `eval()`, the expression is parsed with `ast.parse(..., mode="eval")` and evaluated recursively. Only numeric constants, binary arithmetic operators, and unary +/− are allowed — everything else raises an error.

In [ ]:
_ALLOWED_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
_ALLOWED_UNARYOPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}


def _safe_eval(node: ast.AST) -> float:
    """Recursively evaluate an AST limited to basic arithmetic."""
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        return _ALLOWED_BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported operation: {ast.dump(node)}")


def calculator(expression: str) -> str:
    """A simple calculator that can add, subtract, multiply, or divide two numbers.
    Input should be a mathematical expression like '2 + 2' or '15 / 3'."""
    try:
        tree = ast.parse(expression.strip().strip("'\""), mode="eval")
        result = _safe_eval(tree)
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"The result of {expression.strip()} is {result}"
    except ZeroDivisionError:
        return "Error calculating: division by zero"
    except Exception as e:
        return f"Error calculating: {str(e)}"


# Quick sanity checks
print(calculator("25 + 63"))
print(calculator("15 / 3"))
print(calculator("1 / 0"))
print(calculator("__import__('os')"))  # injection attempt -> rejected

## 3. Tool 2 — Text formatter

Input contract: `'[format_type]: [text]'` where the format type is `uppercase`, `lowercase`, or `titlecase`. The function validates the format type and returns a clear error message otherwise — good error messages matter, because the agent reads them as observations and can self-correct.

In [ ]:
def format_text(text: str) -> str:
    """Format text to uppercase, lowercase, or title case.
    Input should be in format: '[format_type]: [text]'
    where format_type is 'uppercase', 'lowercase', or 'titlecase'."""
    try:
        if ":" not in text:
            return "Error formatting text: input must be '[format_type]: [text]'"
        format_type, _, content = text.partition(":")
        format_type = format_type.strip().strip("'\"").lower()
        content = content.strip().strip("'\"")

        formatters = {
            "uppercase": str.upper,
            "lowercase": str.lower,
            "titlecase": str.title,
        }
        if format_type not in formatters:
            return (
                f"Error formatting text: unknown format '{format_type}'. "
                "Use 'uppercase', 'lowercase', or 'titlecase'."
            )
        return formatters[format_type](content)
    except Exception as e:
        return f"Error formatting text: {str(e)}"


print(format_text("uppercase: hello world"))
print(format_text("titlecase: langchain is awesome"))
print(format_text("reverse: abc"))  # unknown format -> clear error for the agent

## 4. Wrap the functions as `Tool` objects

The `description` is what the LLM actually reads when deciding which tool to call — it is the API contract between your code and the model. Be explicit about the expected input format.

In [ ]:
tools = [
    Tool(
        name="Calculator",
        func=calculator,
        description=(
            "Performs basic arithmetic (+, -, *, /, %, **). "
            "Input: a plain mathematical expression such as '25 + 63' or '15 / 3'."
        ),
    ),
    Tool(
        name="TextFormatter",
        func=format_text,
        description=(
            "Converts text case. Input format: '[format_type]: [text]' where "
            "format_type is 'uppercase', 'lowercase', or 'titlecase'. "
            "Example: 'uppercase: hello world'."
        ),
    ),
]

for t in tools:
    print(f"{t.name}: {t.description}\n")

## 5. The ReAct prompt

`create_react_agent` requires four template variables:

- `{tools}` — auto-filled with each tool's name and description
- `{tool_names}` — comma-separated tool names for the Action line
- `{input}` — the user's question
- `{agent_scratchpad}` — the growing Thought/Action/Observation history across loop iterations

In [ ]:
prompt_template = """You are a helpful assistant who can use tools to help with simple tasks.
You have access to these tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(prompt_template)
print("Template variables:", sorted(prompt.input_variables))

## 6. LLM, agent, and executor

The LLM is an IBM Granite instruct model on **watsonx.ai**.

- Inside the IBM Skills Network lab environment, `project_id="skills-network"` works without an API key.
- Elsewhere, export `WATSONX_APIKEY` and `WATSONX_PROJECT_ID` from your IBM Cloud account first.
- If the model ID is deprecated, substitute a current instruct model from the watsonx catalog.

Two settings matter for agents specifically:

- `stop_sequences=["\nObservation"]` — prevents the model from hallucinating tool outputs instead of waiting for real ones
- `handle_parsing_errors=True` and `max_iterations=5` on the executor — graceful recovery from malformed model output, and a hard stop against infinite loops

In [ ]:
from langchain_ibm import WatsonxLLM

llm = WatsonxLLM(
    model_id="ibm/granite-3-3-8b-instruct",  # verify availability in the watsonx catalog
    url="https://us-south.ml.cloud.ibm.com",
    project_id=os.environ.get("WATSONX_PROJECT_ID", "skills-network"),
    params={
        "decoding_method": "greedy",
        "max_new_tokens": 256,
        "stop_sequences": ["\nObservation"],
    },
)

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,               # print the full ReAct trace
    handle_parsing_errors=True, # recover from malformed LLM output
    max_iterations=5,           # never loop forever
)

print("Agent ready.")

## 7. Run the tests

With `verbose=True`, each run prints the complete reasoning trace — Thought, chosen Action, Action Input, the tool's Observation, and the Final Answer.

In [ ]:
test_questions = [
    "What is 25 + 63?",
    "Can you convert 'hello world' to uppercase?",
    "Calculate 15 * 7",
    "titlecase: langchain is awesome",
]

for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    result = agent_executor.invoke({"input": question})
    print(f"Answer: {result['output']}")

## 8. Conclusion

This exercise demonstrates the anatomy of a LangChain agent:

- **Tools** are plain Python functions wrapped in `Tool` objects — the `description` is the contract the LLM reasons over.
- **The ReAct prompt** structures the reasoning loop; `{agent_scratchpad}` carries state between iterations.
- **`AgentExecutor`** orchestrates the loop, with `stop_sequences`, `handle_parsing_errors`, and `max_iterations` as the difference between a demo and something robust.
- **Security is not optional**: tool inputs come from an LLM, so this solution replaces the suggested `eval()` with a whitelisted AST evaluator.

The repository also includes `test_offline.py` — a scripted `FakeListLLM` test harness that verifies the complete agent loop without requiring watsonx credentials.

## Acknowledgments

Based on Exercise 7 of the IBM Skills Network course *"Build Smarter AI Apps: Empower LLMs with LangChain"*. Starter code and exercise design by the course authors; all solution code written by me.